# Monte Carlo

## Exemplo 1: Estimativa para o valor de π

Um exemplo simples de simulação seria o cálculo de π.

Modelamos um círculo de raio unitário inscrito em um quadrado e geramos pontos aleatórios dentro desse quadrado, que podem assim estar dentro ou fora do círculo. 

Com uma amostragem grande o suficiente de pontos, a razão dos pontos dentro do círculo pelo total de pontos gerados se aproxima de um quarto da razão da área do círculo pela área do quadrado (já que o quadrado terá área igual a 4 unidades de medida de área). 

Assim, estimamos o valor de π numericamente, como demonstrado nos códigos abaixo, utilizando a geração de números pseudo-aleatórios em uma distribuição uniforme entre 0 e 1. Como todos os números gerados no código apresentado são positivos, a área analisada é apenas a do primeiro quadrante. 

Contudo, isso não interfere na razão entre o número de pontos dentro do círculo e o número de pontos dentro do quadrado já que ambos, tanto o círculo quanto o quadrado, têm sua área reduzida em um quarto, mantendo a razão constante.

In [1]:
import numpy as np

In [2]:
def get_pi_using_monte_carlo(N):
    n = 0

    #gerando as coordenadas dos pontos no quadrado unitário seguindo uma distribuição uniforme entre 0 e 1.
    x = np.random.uniform(size=int(N))
    y = np.random.uniform(size=int(N))

    #percorrendo as listas geradas de x e y, se o ponto gerado estiver dentro do primeiro quadrante do círculo, somo 1 no contador n.
    for i in range(int(N)):
        if x[i]**2 + y[i]**2 < 1: 
            n+=1
            
    #como a área de um quadrante do círculo é pi/4, multiplicamos a razão n/N por 4
    pi = 4*n/N

    return pi

In [3]:
def calculate_pi_error(current_value):
    """
    Calculate various error metrics between a current value and the mathematical constant PI.
    
    Parameters:
    -----------
    current_value : float
        The value to compare against PI
    
    Returns:
    --------
    dict
        Dictionary containing multiple error metrics:
        - pi_value: The actual value of PI (np.pi)
        - current_value: The input value for reference
        - absolute_error: |current_value - π|
        - relative_error: (|current_value - π| / π) * 100 (in percentage)
        - squared_error: (current_value - π)²
        - percent_accuracy: 100 - relative_error (how close to 100% accurate)
    
    Examples:
    ---------
    >>> result = calculate_pi_error(3.14)
    >>> print(f"Absolute error: {result['absolute_error']:.6f}")
    Absolute error: 0.001593
    
    >>> result = calculate_pi_error(3.14159265)
    >>> print(f"Relative error: {result['relative_error']:.8f}%")
    Relative error: 0.00000003%
    """
    pi_value = np.pi
    
    # Calculate different error metrics
    absolute_error = abs(current_value - pi_value)
    relative_error = (absolute_error / pi_value) * 100
    squared_error = (current_value - pi_value) ** 2
    percent_accuracy = 100 - relative_error
    
    return {
        'pi_value': pi_value,
        'current_value': current_value,
        'absolute_error': absolute_error,
        'relative_error': relative_error,
        'squared_error': squared_error,
        'percent_accuracy': percent_accuracy
    }

In [4]:
get_pi_using_monte_carlo(1e3)

3.108

In [5]:
get_pi_using_monte_carlo(1e6)

3.141592

In [6]:
pi_val = get_pi_using_monte_carlo(1e9)

In [7]:
print(pi_val)

3.141596336


In [8]:
calculate_pi_error(pi_val)

{'pi_value': 3.141592653589793,
 'current_value': 3.141596336,
 'absolute_error': 3.6824102069843434e-06,
 'relative_error': 0.00011721475738672154,
 'squared_error': 1.3560144932502475e-11,
 'percent_accuracy': 99.99988278524262}

# Monte Carlo Aplicado a Reinforcement Learning

Os métodos de Monte Carlo são formas de estimar funções de valor por meio de experiências (sequência amostral de estados, ações e recompensas). 

Lembre-se de que o valor de um estado é o retorno esperado (recompensa descontada acumulada futura esperada) a partir desse estado. Uma maneira de estimar o retorno esperado é simplesmente calcular a média dos retornos observados após visitas a esse estado. À medida que mais retornos são observados, pela lei dos grandes números, a média deve convergir para o retorno esperado. Essa ideia está na base de todos os métodos Monte Carlo.

Vamos considerar os métodos Monte Carlo para aproximar as políticas ótimas π*.

## On-policy Monte Carlo

![image.png](../images/monte-carlo-on-policy.png)

1. O Monte Carlo on-policy utiliza uma política ε-soft. Uma política soft refere-se a qualquer política que tenha uma probabilidade pequena, mas finita, de selecionar qualquer ação possível, garantindo a exploração de ações alternativas.
2. Após cada episódio, os retornos observados são utilizados para aprender a função de valor da ação e, em seguida, a política é melhorada com base na função de valor aprendida para todos os estados visitados no episódio.
 
No MC on-policy, nossa política desempenha duas funções: gerar trajetórias por meio da exploração e aprender a política ideal. O MC on-policy é um compromisso, pois aprende valores de ação não para a política ideal, mas a partir de uma política quase ideal que ainda explora.



## Off-policy Monte Carlo

O MC off-policy usa duas políticas: uma para aprender a política ideal, chamada política-alvo, e outra para exploração e geração de trajetórias, chamada política de comportamento. Ele segue a política de comportamento enquanto aprende e aprimora a política-alvo.

O algoritmo completo:

![image.png](../images/monte-carlo-off-policy.png)

1. O Monte Carlo off-policy usa a política b, que pode ser qualquer política ε-soft, como política de comportamento, para explorar e gerar trajetórias. Observe que desejamos estimar os retornos esperados sob a política-alvo π, mas tudo o que temos são os retornos da política de comportamento b, o que nos dá uma expectativa errada. Por meio da amostragem por importância, podemos estimar os valores esperados sob uma distribuição, dados os amostras de outra.
2. O algoritmo usa amostragem por importância ponderada, que calcula uma média ponderada dos retornos de acordo com a probabilidade relativa de suas trajetórias ocorrerem sob as políticas-alvo e de comportamento.

Com os métodos de Monte Carlo, é preciso esperar até o final de um episódio antes de atualizar os valores. Se os episódios forem longos, o aprendizado pode se tornar lento.

Reference: https://medium.com/@hsinhungw/intro-to-reinforcement-learning-monte-carlo-to-policy-gradient-1c7ede4eed6e

In [2]:
import sys
from pathlib import Path

# Add parent directory to Python path so we can import modules
parent_dir = Path().resolve().parent
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

In [3]:
import gymnasium as gym

In [4]:
from explorer_strategies import EpsilonGreedy
from learning_techniques import MonteCarlo

epsilon = 0.9
epsilon_greedy_explorer = EpsilonGreedy(epsilon)

# learning rate
lr = 0.9
gamma = 0.9

# Off Policy - usa epsilon-greedy como estratégia de exploration
agent = MonteCarlo(lr, gamma, explorer=epsilon_greedy_explorer)

In [10]:
# Cria ambiente FrozenLake
# is_slippery=False: Ações são determinísticas
env = gym.make('FrozenLake-v1', map_name='4x4', is_slippery=False)

print("Descrição do ambiente:")
print("- Estados: 16 posições na grade 4x4")
print("- Ações: 4 (esquerda, baixo, direita, cima)")
print("- Episódio terminal: Cair em buraco (H) ou atingir objetivo (G)")
print("- Recompensa: 1.0 ao atingir objetivo, 0.0 caso contrário")
print()

Descrição do ambiente:
- Estados: 16 posições na grade 4x4
- Ações: 4 (esquerda, baixo, direita, cima)
- Episódio terminal: Cair em buraco (H) ou atingir objetivo (G)
- Recompensa: 1.0 ao atingir objetivo, 0.0 caso contrário



In [11]:
from params import TrainingConfig

params = TrainingConfig(gamma=gamma, learning_rate=lr, total_episodes=500)

In [12]:
agent.train(env, params)

(array([[0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
      

In [13]:
agent.save_qtable()

Saving Q-table to montecarlo_qtable_4x4.npy


In [15]:
env = gym.make('FrozenLake-v1', map_name='4x4', is_slippery=False, render_mode="human")

observations, info = env.reset()

done = False

while done is False:
    action = agent.act(observations)
    print("action:", action)

    observations, reward, terminated, truncated, info = env.step(action)

    env.render()
    
    done = terminated or truncated

action: 1
action: 1
action: 2
action: 1
action: 2
action: 2
